In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
import mlflow.catboost
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
mlflow.set_tracking_uri("sqlite:///mlflow.db")
import joblib

# Baseline

In [3]:
# Données
df = pd.read_csv("../data/01_raw/Churn_Modelling.csv")
df_clean = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = ['Geography', 'Gender']

# Modèle Baseline
params_baseline = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3,
    'class_weights': {0: 1, 1: 4},
    'cat_features': cat_features,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 0
}

model = CatBoostClassifier(**params_baseline)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

joblib.dump(model, "../model.pkl")
print("Modèle sauvegardé : model.pkl")

# MLflow 
# TRACKING — définir l'experiment
mlflow.set_experiment(  "customer-churn-platform")

with mlflow.start_run(run_name="CatBoost_Baseline"):

    # TRACKING — paramètres
    params_to_log = {
        "modele":         "CatBoost",
        "approche":       "baseline",
        "iterations":     1000,
        "learning_rate":  0.05,
        "depth":          6,
        "l2_leaf_reg":    3,
        "class_weight_minority":4
    }
    mlflow.log_params(params_to_log)

    # TRACKING — métriques
    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "roc_auc":   roc_auc_score(y_test, y_proba),
        "recall":    recall_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred)
    }
    mlflow.log_metrics(metrics)

    # MODELS — sauvegarder le modèle
    mlflow.catboost.log_model(model, "model_baseline")

    # TRACKING — sauvegarder les graphiques SHAP
    mlflow.log_artifact("../data/08_reporting/shap_summary.png")
    mlflow.log_artifact("../data/08_reporting/shap_barplot.png")
    mlflow.log_artifact("../data/08_reporting/shap_waterfall_ahmed.png")
    mlflow.log_artifact("../data/08_reporting/shap_waterfall_sophie.png")

    print(f"ROC-AUC   : {metrics['roc_auc']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"F1        : {metrics['f1']:.4f}")
    print("\n TRACKING - Run enregistré ")
    print(" MODELS Modèle - sauvegardé ")

Modèle sauvegardé : model.pkl


2026/08/17 17:02:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


ROC-AUC   : 0.8536
Recall    : 0.6904
Precision : 0.5467
F1        : 0.6102

 TRACKING - Run enregistré 
 MODELS Modèle - sauvegardé 


# Optuna AUC

In [15]:
# Données
df = pd.read_csv("../data/01_raw/Churn_Modelling.csv")
df_clean = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = ['Geography', 'Gender']

# Modèle CatBoost_Optuna_AUC
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'cat_features': cat_features,
        'eval_metric': 'AUC',
        'random_seed': 42,
        'verbose': 0
    }

    # Cross-validation manuelle
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []

    for train_idx, val_idx in kf.split(X_train, y_train):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(**params) 
        model.fit(X_tr, y_tr)

        y_proba = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_proba)
        auc_scores.append(auc)

    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30,
               show_progress_bar=True)

# Après 
best_params_final = study.best_params.copy()
best_params_final['cat_features'] = cat_features
best_params_final['eval_metric'] = 'AUC'
best_params_final['random_seed'] = 42
best_params_final['verbose'] = 0

model_optimized = CatBoostClassifier(**best_params_final)
model_optimized.fit(X_train, y_train)

y_pred  = model_optimized.predict(X_test)
y_proba = model_optimized.predict_proba(X_test)[:, 1]

# MLflow 
# TRACKING — définir l'experiment

mlflow.set_experiment(  "customer-churn-platform")

with mlflow.start_run(run_name="CatBoost_Optuna_AUC"):

    # TRACKING — paramètres
    best_params = study.best_params

    params_to_log = {
    "modele":                "CatBoost",
    "approche":              "Optuna_AUC",
    "iterations":            best_params['iterations'],
    "learning_rate":         best_params['learning_rate'],
    "depth":                 best_params['depth'],
    "l2_leaf_reg":           best_params['l2_leaf_reg'],
    "class_weight_minority": 4
}
    mlflow.log_params(params_to_log)

    # TRACKING — métriques
    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "roc_auc":   roc_auc_score(y_test, y_proba),
        "recall":    recall_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred)
    }
    mlflow.log_metrics(metrics)

    # MODELS — sauvegarder le modèle
    mlflow.catboost.log_model(model_optimized, "model_Optuna_AUC")
    # Pas d'artefact ici 

    print(f"ROC-AUC   : {metrics['roc_auc']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"F1        : {metrics['f1']:.4f}")
    print("\n TRACKING - Run enregistré ")
    print(" MODELS Modèle - sauvegardé ")

  0%|          | 0/30 [00:00<?, ?it/s]

2026/08/17 03:37:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


ROC-AUC   : 0.8749
Recall    : 0.4619
Precision : 0.8103
F1        : 0.5884

 TRACKING - Run enregistré 
 MODELS Modèle - sauvegardé 


# Optuna Recall

In [16]:
# Données
df = pd.read_csv("../data/01_raw/Churn_Modelling.csv")
df_clean = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_features = ['Geography', 'Gender']

# CatBoost_Optuna_Recall
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        # Laisser Optuna trouver le bon poids pour la classe minoritaire
        'class_weights': {
            0: 1,
            1: trial.suggest_int('class_weight_minority', 2, 8)
        },
        'cat_features': cat_features,
        'eval_metric': 'AUC',
        'random_seed': 42,
        'verbose': 0
    }

    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    recall_scores = []

    for train_idx, val_idx in kf.split(X_train, y_train):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)
        recall = recall_score(y_val, y_pred)
        recall_scores.append(recall)

    return np.mean(recall_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30,
               show_progress_bar=True)

print(f"\nMeilleur Recall : {study.best_value:.4f}")
print(f"Meilleurs paramètres : {study.best_params}")

# Après 
best_params = study.best_params.copy()
class_weight_minority = best_params['class_weight_minority']

best_params.pop('class_weight_minority')

best_params['class_weights'] = {0: 1, 1: class_weight_minority}
best_params['cat_features'] = cat_features
best_params['eval_metric'] = 'AUC'
best_params['random_seed'] = 42
best_params['verbose'] = 100


model_optimized = CatBoostClassifier(**best_params)
model_optimized.fit(X_train, y_train)

y_pred  = model_optimized.predict(X_test)
y_proba = model_optimized.predict_proba(X_test)[:, 1]

# MLflow 
# TRACKING — définir l'experiment

mlflow.set_experiment(  "customer-churn-platform")

with mlflow.start_run(run_name="CatBoost_Optuna_Recall"):

    # TRACKING — paramètres
    best_params = study.best_params

    params_to_log = {
    "modele":                "CatBoost",
    "approche":              "Optuna_Recall",
    "iterations":            best_params['iterations'],
    "learning_rate":         best_params['learning_rate'],
    "depth":                 best_params['depth'],
    "l2_leaf_reg":           best_params['l2_leaf_reg'],
    "class_weight_minority": best_params['class_weight_minority']
}
    mlflow.log_params(params_to_log)

    # TRACKING — métriques
    metrics = {
        "accuracy":  accuracy_score(y_test, y_pred),
        "roc_auc":   roc_auc_score(y_test, y_proba),
        "recall":    recall_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "f1":        f1_score(y_test, y_pred)
    }
    mlflow.log_metrics(metrics)

    # MODELS — sauvegarder le modèle
    mlflow.catboost.log_model(model_optimized, "model_Optuna_Recall")
    # Pas d'artefact ici 

    print(f"ROC-AUC   : {metrics['roc_auc']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"F1        : {metrics['f1']:.4f}")
    print("\n TRACKING - Run enregistré ")
    print(" MODELS Modèle - sauvegardé ")

  0%|          | 0/30 [00:00<?, ?it/s]


Meilleur Recall : 0.8963
Meilleurs paramètres : {'iterations': 172, 'learning_rate': 0.0116740732454461, 'depth': 9, 'l2_leaf_reg': 3.468828109128747, 'class_weight_minority': 8}
0:	total: 4.02ms	remaining: 687ms
100:	total: 422ms	remaining: 296ms


2026/08/17 03:40:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


171:	total: 708ms	remaining: 0us
ROC-AUC   : 0.8532
Recall    : 0.8968
Precision : 0.3434
F1        : 0.4966

 TRACKING - Run enregistré 
 MODELS Modèle - sauvegardé 
